<a href="https://colab.research.google.com/github/martinaditrani/googletest/blob/main/colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NeuralNets CUDA (Colab)

This notebook compiles and runs the GPU part of the project on a Colab GPU.
You **edit code locally in VS Code, push to GitHub, and just re-run this notebook** — Colab is only used as a compiler + GPU.

**Before running:** `Runtime → Change runtime type → Hardware accelerator: GPU`.

Cells 1–4 are the Milestone 0 smoke test; cell 5 runs the Milestone 1 matmul benchmark.

## 1. Check the GPU and CUDA toolkit
Colab ships `nvcc` and a GPU driver. This confirms both are present.

In [ ]:
!nvidia-smi
!nvcc --version
!cmake --version

/bin/bash: line 1: nvidia-smi: command not found
/bin/bash: line 1: nvcc: command not found
cmake version 3.31.10

CMake suite maintained and supported by Kitware (kitware.com/cmake).


## 2. Get the code (private repo)

The repo is **private**, so plain `git clone` over HTTPS fails with
`could not read Username`. We authenticate with a **GitHub Personal Access
Token (PAT)**, stored in Colab Secrets so it never gets written into the
notebook.

**One-time setup:**
1. GitHub → Settings → Developer settings → *Personal access tokens* →
   *Fine-grained tokens* → Generate. Give it **read-only `Contents`** access
   to this repo only.
2. In Colab, click the **🔑 key icon** in the left sidebar (Secrets).
3. Add a secret named `GH_TOKEN`, paste the token as its value, and enable
   *Notebook access*.

First run: clone. Later runs: `git pull` to fetch your latest pushes.

In [ ]:
import os
from google.colab import drive

# Mount your Google Drive (This will prompt you to authorize once per session)
drive.mount('/content/drive')

REPO_SLUG = 'Sckun16/my-neuralnets-2-neuralnets'
REPO_DIR = '/content/neuralnets'

# Read the token from your secure Google Drive file
token_path = '/content/drive/MyDrive/github_token.txt'
with open(token_path, 'r') as file:
    token = file.read().strip()

# Embed the token in the URL
auth_url = f'https://{token}@github.com/{REPO_SLUG}.git'

if not os.path.isdir(REPO_DIR):
    !git clone $auth_url $REPO_DIR
else:
    !cd $REPO_DIR && git pull

%cd $REPO_DIR

Mounted at /content/drive
Cloning into '/content/neuralnets'...
remote: Enumerating objects: 315, done.
remote: Counting objects: 100% (62/62), done.
remote: Compressing objects: 100% (47/47), done.
remote: Total 315 (delta 24), reused 33 (delta 14), pack-reused 253 (from 1)
Receiving objects: 100% (315/315), 325.24 KiB | 925.00 KiB/s, done.
Resolving deltas: 100% (141/141), done.
/content/neuralnets


## 3. Build the CUDA target
Uses the standalone `cuda/CMakeLists.txt`, which needs **only** the CUDA toolkit
(no OpenBLAS/LAPACK/OpenMP). `CMAKE_CUDA_ARCHITECTURES` defaults to 75 (T4);
override it if Colab gives you a different GPU (e.g. 80 for A100).

In [ ]:
!cmake -S cuda -B build-cuda -DCMAKE_BUILD_TYPE=Release
!cmake --build build-cuda -j

-- The CXX compiler identification is GNU 11.4.0
CMake Error at /usr/local/lib/python3.12/dist-packages/cmake/data/share/cmake-3.31/Modules/Internal/CMakeCUDAFindToolkit.cmake:104 (message):
  Failed to find nvcc.

  Compiler requires the CUDA toolkit.  Please set the CUDAToolkit_ROOT
  variable.
Call Stack (most recent call first):
  /usr/local/lib/python3.12/dist-packages/cmake/data/share/cmake-3.31/Modules/CMakeDetermineCUDACompiler.cmake:85 (cmake_cuda_find_toolkit)
  CMakeLists.txt:18 (project)


-- Configuring incomplete, errors occurred!
gmake: Makefile: No such file or directory
gmake: *** No rule to make target 'Makefile'.  Stop.


## 4. Run the device query + smoke test
Expected: device properties printed and `Kernel round-trip self-test: PASSED`.

In [ ]:
!./build-cuda/hello_cuda

/bin/bash: line 1: ./build-cuda/hello_cuda: No such file or directory


## 5. Milestone 1 — naive matmul vs cuBLAS

Times the hand-written naive kernel against cuBLAS (the GPU's "OpenBLAS") and
verifies correctness. cuBLAS will be far faster — that gap is the whole point;
later milestones (tiling, register blocking) close it.

Args: `--m --n --k` (sizes), `--type=float|double`, `--iters=N`, `--check=1`.

In [ ]:
# Default run: 1024^3 float, verified against cuBLAS
!./build-cuda/benchmark_cuda --m=1024 --n=1024 --k=1024 --type=float --check=1

# A size sweep (uncomment to gather data for the GFLOPS-vs-size plot):
# for N in 256 512 1024 2048 4096:
#     !./build-cuda/benchmark_cuda --m=$N --n=$N --k=$N --type=float --check=0

/bin/bash: line 1: ./build-cuda/benchmark_cuda: No such file or directory


## 6. Milestone 2 — tiled matmul vs cuBLAS & naive

Times the hand-written tiled kernel (using shared memory) against both the naive baseline and cuBLAS. Tiling should significantly increase performance by reusing blocks of data in fast on-chip memory, shrinking the gap toward the vendor BLAS ceiling.

Args: `--m --n --k` (sizes), `--type=float|double`, `--iters=N`, `--check=1`.

In [ ]:
# A size sweep (uncomment to gather data for the GFLOPS-vs-size plot):
 for N in 256 512 1024 2048 4096:
     !./build-cuda/benchmark_cuda --m=$N --n=$N --k=$N --type=float --check=0

/bin/bash: line 1: ./build-cuda/benchmark_cuda: No such file or directory
